<div style="border: 5px solid black; padding: 20px; border-radius: 6px;">

## **Course:** DSC670 - Advanced Uses of Generative AI

## **Name:** Tim Hollis

## **Assignment:** Assessment 4.2

## **Date:** June 30, 2026

---

**References**

Bahree, A. (2024). *Generative AI in action*. Manning Publications.

OpenAI. (2024). *OpenAI API documentation*. https://platform.openai.com/docs

</div>

### **Initial Setup**

In [1]:
# Libraries
from openai import OpenAI
from dotenv import load_dotenv
import json
import os
from IPython.display import display, Markdown, HTML

display(HTML('''
<style>
    div.output_subarea { page-break-inside: avoid; }
    div.jp-MarkdownOutput { page-break-inside: avoid; }
    div.cell { page-break-inside: avoid; }
</style>
'''))

# Load environment variables from .env file
load_dotenv()

# OPENAI_API_KEY from environment
client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

print('✅ Setup complete - OpenAI client ready')

✅ Setup complete - OpenAI client ready


## Problem 1: Math and Extraction with Zero and One-Shot Prompting

Both emails describe a shoe order with no listed prices, so part of the extraction task is asking 
the model to assign a reasonable retail price per shoe and calculate subtotals and a grand total 
itself. The zero-shot prompt gives the model only a field definition and no worked example. The 
one-shot prompt reuses the zero-shot output for email 1 as a worked example inside the same message 
history, then asks the model to apply that same pattern to email 2 in a single API call.

In [2]:
# Email 1 provided as raw text for extraction
email_1 = '''Hey Jim,

I'd like to order some new shoes. Please ship the following:

1. Nike Air Jordan I - 2 pair
2. Converse All-Star - 10 pair
3. New Balance 990 - 1 pair
4. Nike Zoom Fly 5 men's red - 2 pair

Please provide sub-totals and grand total cost.

Thanks.

Lance Gentry
123 Main St.
Chelsea, MI 48109
248-229-2229'''

# Zero-shot prompt - field definitions only, no worked example provided
zero_shot_system = '''You are an assistant that extracts structured order information from shoe order emails.
Extract the following fields and return valid JSON only, with no extra commentary:
- shoe_agent_name: the person the email is addressed to
- request_type: the type of request (e.g. "Order")
- items: a list of objects, one per shoe line item, each containing:
    - shoe_brand
    - shoe_model
    - shoe_quantity
    - shoe_price: assign a reasonable current retail price in USD for the shoe
    - shoe_subtotal: shoe_price multiplied by shoe_quantity
- customer_name
- customer_street
- customer_city
- customer_state
- customer_zip
- customer_phone
- grand_total: the sum of all shoe_subtotal values

Return only the JSON object, formatted with 2-space indentation.'''

response_zero_shot = client.chat.completions.create(
    model='gpt-4o',
    messages=[
        {'role': 'system', 'content': zero_shot_system},
        {'role': 'user', 'content': email_1}
    ],
    temperature=0
)

zero_shot_output = response_zero_shot.choices[0].message.content
print('✅ Zero-shot extraction complete for email 1')
print(zero_shot_output)

✅ Zero-shot extraction complete for email 1
```json
{
  "shoe_agent_name": "Jim",
  "request_type": "Order",
  "items": [
    {
      "shoe_brand": "Nike",
      "shoe_model": "Air Jordan I",
      "shoe_quantity": 2,
      "shoe_price": 180,
      "shoe_subtotal": 360
    },
    {
      "shoe_brand": "Converse",
      "shoe_model": "All-Star",
      "shoe_quantity": 10,
      "shoe_price": 60,
      "shoe_subtotal": 600
    },
    {
      "shoe_brand": "New Balance",
      "shoe_model": "990",
      "shoe_quantity": 1,
      "shoe_price": 175,
      "shoe_subtotal": 175
    },
    {
      "shoe_brand": "Nike",
      "shoe_model": "Zoom Fly 5 men's red",
      "shoe_quantity": 2,
      "shoe_price": 160,
      "shoe_subtotal": 320
    }
  ],
  "customer_name": "Lance Gentry",
  "customer_street": "123 Main St.",
  "customer_city": "Chelsea",
  "customer_state": "MI",
  "customer_zip": "48109",
  "customer_phone": "248-229-2229",
  "grand_total": 1455
}
```


In [3]:
email_2 = '''Hey Michelle,

I'd like to order some new shoes. Please ship the following:

1. Nike Air Jordan I - 1 pair
2. Converse All-Star - 20 pair
3. New Balance 990 - 2 pair
4. Nike Zoom Fly 5 men's red - 5 pair

Please provide sub-totals and grand total cost.

Thanks.

Artis Gilmore
723 Lexington Blvd.
New York, NY 10001
(503) 484-1029'''

# One-shot prompt - the zero-shot result for email 1 is included as a worked example
# before the model is asked to extract email 2, all within a single API call
one_shot_system = '''You are an assistant that extracts structured order information from shoe order emails.
Extract these fields and return valid JSON only, with no extra commentary:
shoe_agent_name, request_type, items (list of shoe_brand, shoe_model, shoe_quantity, shoe_price, shoe_subtotal),
customer_name, customer_street, customer_city, customer_state, customer_zip, customer_phone, grand_total.
shoe_price should be a reasonable current retail price in USD you assign for each shoe.
Return only the JSON object, formatted with 2-space indentation.'''

response_one_shot = client.chat.completions.create(
    model='gpt-4o',
    messages=[
        {'role': 'system', 'content': one_shot_system},
        {'role': 'user', 'content': f'Email:\n{email_1}'},
        {'role': 'assistant', 'content': zero_shot_output},
        {'role': 'user', 'content': f'Email:\n{email_2}'}
    ],
    temperature=0
)

one_shot_output = response_one_shot.choices[0].message.content
print('✅ One-shot extraction complete for email 2')
print(one_shot_output)

✅ One-shot extraction complete for email 2
```json
{
  "shoe_agent_name": "Michelle",
  "request_type": "Order",
  "items": [
    {
      "shoe_brand": "Nike",
      "shoe_model": "Air Jordan I",
      "shoe_quantity": 1,
      "shoe_price": 180,
      "shoe_subtotal": 180
    },
    {
      "shoe_brand": "Converse",
      "shoe_model": "All-Star",
      "shoe_quantity": 20,
      "shoe_price": 60,
      "shoe_subtotal": 1200
    },
    {
      "shoe_brand": "New Balance",
      "shoe_model": "990",
      "shoe_quantity": 2,
      "shoe_price": 175,
      "shoe_subtotal": 350
    },
    {
      "shoe_brand": "Nike",
      "shoe_model": "Zoom Fly 5 men's red",
      "shoe_quantity": 5,
      "shoe_price": 160,
      "shoe_subtotal": 800
    }
  ],
  "customer_name": "Artis Gilmore",
  "customer_street": "723 Lexington Blvd.",
  "customer_city": "New York",
  "customer_state": "NY",
  "customer_zip": "10001",
  "customer_phone": "(503) 484-1029",
  "grand_total": 2530
}
```


In [4]:
def clean_json_response(raw_text):
    """Strip markdown code fences from a model response before JSON parsing"""
    cleaned = raw_text.strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.split('\n', 1)[1]  # drop the ```json line
        cleaned = cleaned.rsplit('```', 1)[0]  # drop the trailing ```
    return cleaned.strip()


zero_shot_json = json.loads(clean_json_response(zero_shot_output))
one_shot_json = json.loads(clean_json_response(one_shot_output))

print('✅ Zero-shot output is valid JSON')
print(f'   Grand total: ${zero_shot_json.get("grand_total")}')
print('✅ One-shot output is valid JSON')
print(f'   Grand total: ${one_shot_json.get("grand_total")}')

✅ Zero-shot output is valid JSON
   Grand total: $1455
✅ One-shot output is valid JSON
   Grand total: $2530


### Problem 1 Summary

Both prompts extracted every field correctly and the math checked out on both emails once I 
manually verified the totals, `$1,455` for email 1 and `$2,530` for email 2. The real difference between 
the two approaches was not accuracy, it was consistency of judgment calls the model had to make on 
its own. Neither email listed a price for any shoe, so the model had to invent reasonable retail 
prices, and this is where the one-shot approach earned its keep. In the zero-shot prompt, the model 
set its own prices for email 1 with no example to anchor against. In the one-shot prompt, I fed that 
same zero-shot result back in as a prior turn before asking it to extract email 2, and the model 
reused the exact same price per shoe brand and model across both emails, `$180` for the Air Jordan I, 
`$60` for the All-Star, `$175` for the 990, and `$160` for the Zoom Fly 5. That consistency was not 
guaranteed. Without the worked example, there was nothing stopping the model from pricing the same 
shoe differently on a second, independent call, which would have made the two orders impossible to 
compare or reconcile against each other in a real order system. The one-shot example effectively 
locked in a price list for the rest of the conversation.

The structural fields, names, addresses, phone numbers, and quantities, were extracted correctly in 
both prompts with no meaningful difference, which tells me the model did not need an example to 
handle that part of the task. The value of one-shot prompting here was narrow but real: it showed up 
specifically where the task required the model to invent information rather than just extract it, 
and having a prior example gave it something consistent to invent from. The one practical issue in 
both runs was that the model wrapped its JSON output in markdown code fences despite the system 
prompt explicitly saying to return the JSON object only, which broke the first parsing attempt and 
required a small cleanup step before `json.loads()` would accept it. That is a good reminder that 
telling a model what not to include does not always work as reliably as constraining the output 
format more directly, for example by using a JSON mode or response schema instead of a plain 
instruction.

## Problem 2: Chat Completion for Chain of Thought

This recreates the chain-of-thought example from Listing 6.4 in *Generative AI in Action* (Bahree, 
2024), using the OpenAI API directly instead of Azure OpenAI. The three worked Q&A pairs from the 
book are provided as prior turns in the message history, which primes the model to show its 
step-by-step reasoning before stating a final answer, rather than jumping straight to a number. The 
fourth question, the age riddle, is then asked as a new user turn in the same conversation.

In [5]:
# Chain-of-thought example messages, recreating Listing 6.4 (Bahree, 2024)
# via the OpenAI API
cot_messages = [
    {'role': 'user',
     'content': 'There were nine computers in the server room. Five more '
                'computers were installed each day, from Monday to Thursday. '
                'How many computers are now in the server room?'},
    {'role': 'assistant',
     'content': 'There are 4 days from Monday to Thursday. 5 computers were '
                'added each day. That means in total 4 * 5 = 20 computers '
                'were added. There were 9 computers initially, so now there '
                'are 9 + 20 = 29 computers. The answer is 29.'},
    {'role': 'user',
     'content': 'Michael had 58 golf balls. On Tuesday, he lost 23 golf '
                'balls. On Wednesday, he lost 2 more. How many golf balls '
                'did he have at the end of Wednesday?'},
    {'role': 'assistant',
     'content': 'Michael initially had 58 balls. He lost 23 on Tuesday, so '
                'after that he has 58 - 23 = 35 balls. On Wednesday he lost '
                '2 more so now he has 35 - 2 = 33 balls. The answer is 33.'},
    {'role': 'user',
     'content': 'Olivia has $23. She bought five bagels for $3 each. How '
                'much money does she have left?'},
    {'role': 'assistant',
     'content': 'She bought 5 bagels for $3 each. This means she spent $15. '
                'She has $8 left.'},
    {'role': 'user',
     'content': "When I was 6 my sister was half my age. Now I'm 70 how old "
                'is my sister?'}
]

response_cot = client.chat.completions.create(
    model='gpt-4o',
    messages=cot_messages,
    temperature=0
)

cot_answer = response_cot.choices[0].message.content
print('✅ Chain-of-thought completion generated')
print(cot_answer)

✅ Chain-of-thought completion generated
When you were 6, your sister was half your age, which means she was 3 years old at that time. The age difference between you and your sister is 6 - 3 = 3 years. Now that you are 70, your sister would be 70 - 3 = 67 years old.


<h2 style="margin-top:0.75in;">Problem 2 Summary</h2>

The three worked examples from the textbook and the new question I asked are different in one 
important way: the first three are pure arithmetic with no trick to catch, while the age question is 
a classic riddle where the surface-level operation, dividing 70 by 2, produces a plausible-looking 
but wrong answer of 35. The server room question and the golf ball question both just required 
tracking a running total through a sequence of additions and subtractions, and the model's replies 
mirrored the style of the worked examples almost exactly, stating the operation, showing the 
arithmetic, and closing with "The answer is X." The bagel question was slightly different since the 
book's own answer skipped showing the subtraction step and just stated the result, and that shorter, 
less explicit style in the third example seemed to influence how compact the model's own reasoning 
style could have been if the pattern alone was driving the response.

The age question is where the comparison gets interesting. If the model had simply pattern-matched 
the arithmetic style of the previous three answers without actually reasoning about the relationship 
between the ages, I would have expected something like "half of 70 is 35, the answer is 35," treating 
"half my age" as if it applied at both points in time. Instead, the response correctly identified 
that the sister was 3 years old when I was 6, calculated the fixed 3-year age gap, and applied that 
gap to my current age of 70 to get 67. That is a genuinely different kind of reasoning than the first 
three questions required, since it depends on recognizing that an age difference stays constant over 
time even though a ratio like "half my age" does not. The chain-of-thought priming from the three 
worked examples seems to have encouraged the model to write out its steps explicitly rather than 
jump straight to an answer, and writing out the steps is likely what kept it from defaulting to the 
tempting but incorrect 35. Whether that would hold up on a genuinely novel trick question with a less 
common structure is a fair thing to be skeptical about, but for this one, showing its work appears to 
have been the difference between getting the riddle right and falling into the obvious trap.

## **Conclusion**

This exercise put zero-shot, one-shot, and chain-of-thought prompting side by side, and the pattern 
that stood out across all of it was that examples do their best work when a task requires judgment 
rather than pure extraction or arithmetic. In Problem 1, the model extracted names, addresses, and 
quantities correctly whether or not it had a worked example to follow, but it only held a consistent 
set of invented shoe prices across both emails once the one-shot example gave it something to anchor 
to. In Problem 2, the three worked examples were not there to teach the model addition and 
subtraction, they were there to establish a habit of writing out reasoning step by step before 
answering, and that habit is what appears to have carried over successfully to a genuinely different 
kind of problem, the age riddle, where the obvious shortcut answer was wrong. Across both problems, 
the model was consistently better at literal extraction and simple arithmetic than at exercising 
judgment or catching a trick, which lines up with what I would expect: giving a model structure to 
imitate helps most exactly where the task stops being mechanical. The one practical lesson worth 
carrying forward is that telling a model what not to do, like avoiding markdown code fences around 
JSON, is a weaker constraint than showing it the exact output format through an example or a stricter 
response mode, since instructions alone did not reliably prevent that formatting issue in either 
extraction run.

# Reflection

This week brought together two major pieces of the course: Assignment 4.2’s prompt‑based API work and Milestone 2’s deeper design and experimentation for the fraudulent job posting project. Working through both assignments made the connection between prompting techniques and model judgment much clearer. Assignment 4.2 showed how zero‑shot, one‑shot, and chain‑of‑thought prompting behave in controlled tasks, while Milestone 2 demonstrated how those same techniques scale — and where they fail — when applied to a real problem like job fraud detection. Seeing both side by side helped me understand why prompting alone can shape structure but not reliably shape judgment.

## Straightforward Aspects

- The extraction tasks in Problem #1 were smooth because the structure of the emails made the fields easy to isolate. The model handled names, addresses, quantities, and phone numbers consistently, and as I noted in my write‑up, “Both prompts extracted every field correctly and the math checked out on both emails once I manually verified the totals.”  
- Recreating the chain‑of‑thought example from the textbook was also straightforward. The model followed the pattern of the worked examples and produced a correct, step‑by‑step answer to the age riddle, writing: “When you were 6, your sister was half your age… Now that you are 70, your sister would be 70 - 3 = 67 years old.”  
- Structuring Milestone 2 was clear because the APA format naturally separated the updates to the problem statement, the model choice, the prompt experiments, and the fine‑tuning plan. The narrative about treating verdict + explanation as a single generation task came together cleanly.

## Challenging Aspects

- The biggest challenge in Assignment 4.2 was seeing how differently the model behaves when it has to invent information versus extract it. The one‑shot prompt “effectively locked in a price list for the rest of the conversation,” but it also highlighted how fragile invented values are when the example changes.  
- In Milestone 2, the prompt experiments were challenging because none of them solved the judgment problem. As I wrote, “Prompting redistributed the errors rather than eliminating them. No single prompt gave this model both the suspicion to catch disguised fraud and the fairness to clear unusual but legitimate postings.” Watching different strategies trade false negatives for false positives forced me to think more seriously about recall‑first design.  
- Reframing fine‑tuning as judgment‑building instead of capability‑building was conceptually difficult. It means the training data has to encode calibrated suspicion, not just labels, and I had to confront the risk that “a fluent rationale attached to a wrong verdict is more dangerous to a job seeker than no tool at all.” That raised the bar on explanation quality and made data construction feel like a responsibility rather than a mechanical step.

